In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import colors
import os
import re
from tkinter import Tk, filedialog
from moviepy.editor import ImageSequenceClip
from matplotlib.patches import Rectangle

# --- Domain & time settings ---
X_MIN, X_MAX = 400, 1400
Y_MIN, Y_MAX = 0, 400
START_TIME = 40
END_TIME = 110
TIME_STEP = 1
SECONDS_PER_FRAME = 0.3

# --- Colormap range ---
Z_MIN, Z_MAX = 16, 25

# --- Figure layout (inches) ---
AXIS_WIDTH_INCH = 5
AXIS_HEIGHT_INCH = 1.6
LEFT_MARGIN_INCH = 0.5
BOTTOM_MARGIN_INCH = 0.5

plt.style.use('default')
plt.rcParams.update({
    'font.family': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 6,
    'axes.titlesize': 7,
    'axes.labelsize': 6,
    'xtick.labelsize': 5,
    'ytick.labelsize': 5,
    'figure.dpi': 300,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5
})


def pick_file(title):
    r = Tk()
    r.withdraw()
    r.update()
    f = filedialog.askopenfilename(title=title)
    r.destroy()
    return f


def parse_time_blocks(file):
    with open(file) as f:
        lines = f.readlines()
    t, vals = None, []
    times, blocks = [], []
    for l in lines:
        if "time=" in l:
            if t is not None:
                times.append(t)
                blocks.append(np.array(vals))
            t = float(re.findall(r"[-+]?\d*\.\d+|\d+", l)[0])
            vals = []
        else:
            try:
                vals.append(float(l.strip()))
            except:
                pass
    if t is not None:
        times.append(t)
        blocks.append(np.array(vals))
    return np.array(times), blocks


def parse_xy_grid(file):
    with open(file) as f:
        lines = f.readlines()
    t, xs, ys = None, [], []
    times, xb, yb = [], [], []
    for l in lines:
        l = l.strip()
        if "time=" in l:
            if t is not None:
                times.append(t)
                xb.append(np.array(xs))
                yb.append(np.array(ys))
            t = float(re.findall(r"[-+]?\d*\.\d+|\d+", l)[0])
            xs, ys = [], []
        else:
            try:
                x, y = l.split(",")
                xs.append(float(x))
                ys.append(float(y))
            except:
                pass
    if t is not None:
        times.append(t)
        xb.append(np.array(xs))
        yb.append(np.array(ys))
    return np.array(times), xb, yb


def nearest_index(times, t):
    return np.argmin(np.abs(times - t))


def plot_frame(x, y, z, t, out_path):
    n = min(len(x), len(y), len(z))
    x, y, z = x[:n], y[:n], z[:n]

    x = x / 1000.0
    y = y / 1000.0

    BUF = 200
    mask = (
        np.isfinite(x) & np.isfinite(y) & np.isfinite(z) &
        (x >= X_MIN - BUF) & (x <= X_MAX + BUF) &
        (y >= Y_MIN - BUF) & (y <= Y_MAX + BUF)
    )
    x, y, z = x[mask], y[mask], z[mask]

    if len(x) < 10:
        print(f"Skipping t={t:.2f}: only {len(x)} points in domain.")
        return False

    print(f"  t={t:.2f}: {len(x)} pts | x=[{x.min():.0f}, {x.max():.0f}] | y=[{y.min():.0f}, {y.max():.0f}] | z=[{z.min():.1f}, {z.max():.1f}]")

    visc_cmap = colors.LinearSegmentedColormap.from_list(
        'visc_cmap',
        [
            '#FFFF00', '#FFEB00', '#FFD700',
            '#ADFF2F', '#7CFC00',
            '#40E0D0', '#00BFFF',
            '#9370DB', '#8A2BE2', '#9400D3'
        ]
    )

    fig_width = AXIS_WIDTH_INCH + LEFT_MARGIN_INCH + 0.2
    fig_height = AXIS_HEIGHT_INCH + BOTTOM_MARGIN_INCH + 0.2
    fig = plt.figure(figsize=(fig_width, fig_height))
    ax = fig.add_axes([
        LEFT_MARGIN_INCH / fig_width,
        BOTTOM_MARGIN_INCH / fig_height,
        AXIS_WIDTH_INCH / fig_width,
        AXIS_HEIGHT_INCH / fig_height
    ])

    norm = colors.Normalize(vmin=Z_MIN, vmax=Z_MAX)
    im = ax.tricontourf(
        x, y, z,
        levels=np.linspace(Z_MIN, Z_MAX, 256),
        cmap=visc_cmap,
        norm=norm,
        extend='both'
    )

    clip_rect = Rectangle((X_MIN, Y_MIN), X_MAX - X_MIN, Y_MAX - Y_MIN,
                           transform=ax.transData)
    for col in im.collections:
        col.set_clip_path(clip_rect)

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MAX, Y_MIN)
    ax.set_xlabel('Distance, km', fontsize=6)
    ax.set_ylabel('Depth, km', fontsize=6)
    ax.tick_params(labelsize=5)

    cbar = fig.colorbar(im, ax=ax, fraction=0.12, pad=0.005)
    cbar.set_label('Viscosity, Pa·s', fontsize=6)
    cbar.ax.tick_params(labelsize=5)
    cbar.outline.set_linewidth(0.5)
    ticks = np.linspace(Z_MIN, Z_MAX, 10)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels([str(int(tk)) for tk in ticks])

    ax.text(
        0.98, 0.03,
        f"{int(t // 10) * 10} Myr",
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=6,
        bbox=dict(facecolor="white", edgecolor="black", linewidth=0.5, pad=0.15)
    )

    plt.savefig(out_path, dpi=300)
    plt.close('all')
    return True


def main():
    print("Select input files...")
    vf = pick_file("Select viscosity file")
    xy = pick_file("Select XY grid file")

    out_dir = os.path.dirname(vf)
    os.makedirs(out_dir, exist_ok=True)

    t_v, v_blocks = parse_time_blocks(vf)
    t_xy, x_blocks, y_blocks = parse_xy_grid(xy)

    frames = []
    for i in range(len(t_v)):
        if i % TIME_STEP:
            continue
        t = t_v[i]
        if t < START_TIME or t > END_TIME:
            continue
        ix = nearest_index(t_xy, t)
        out = os.path.join(out_dir, f"frame_{i:04d}.png")
        if plot_frame(x_blocks[ix], y_blocks[ix], v_blocks[i], t, out):
            frames.append(out)
            print(f"  Saved frame {i} (t={t:.2f})")

    if not frames:
        print("No frames were generated. Check time range and input files.")
        return

    print(f"\nRendering video from {len(frames)} frames...")
    fps = 1 / SECONDS_PER_FRAME
    clip = ImageSequenceClip(frames, fps=fps)
    clip.write_videofile(
        os.path.join(out_dir, "visc.mp4"),
        codec="libx264",
        fps=fps,
        audio=False
    )
    print("Done. Output saved to visc.mp4")


if __name__ == "__main__":
    main()